# 🚀 Google AX (Agentic Orchestration Runtime) 完整實戰教學

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Child-pi/ax/blob/main/examples/ax_google_colab_demo.ipynb)

本筆記本帶您深入探索與實際操作 **Google AX**（[Child-pi/ax](https://github.com/Child-pi/ax) / [google/ax](https://github.com/google/ax)）。

### 📖 什麼是 AX？
**AX** 是 Google 開發的高通量、宣告式自主代理（Autonomous Agent）編排平台，專為在叢集中運行數十億個沙盒化 Agent 任務而設計。

#### 四大核心物件（CRD-like Primitives）：
1. **`Task`**：Agent 的最小隔離執行單元（資源限制、容器命令、環境變數、暫停/恢復）。
2. **`Workspace`**：預先配載 Git 倉庫、MCP Servers、Skills，支援由 AI 依據自然語言 `goal` 自動建置環境。
3. **`Gateway`**：沙盒網路邊界圍欄，定義對外連線白名單（Egress Allowlist）與監聽埠。
4. **`Model`**：叢集層級的 LLM 統一配置（支援 Gemini、Claude 等模型與憑證參照）。

---
### 🛠 本筆記本包含以下實作步驟：
1. **環境配置**：安裝 Go 1.23+、Redis 伺服器與相關依賴
2. **下載與編譯**：編譯 `ax` (CLI)、`ax-server` (Control Plane API)、`ax-task-runner` (沙盒執行器)
3. **啟動控制平面**：啟動 Redis 與 AX Server
4. **宣告式任務部署**：使用 `ax` CLI 管理 Task、Workspace、Gateway、Model
5. **沙盒執行器實測**：模擬沙盒環境啟動、Workspace Git 自動拉取與 Metadata API 查詢
6. **生命週期控制**：狀態檢查、Describe 詳細資訊與資源清理

## 步驟 1：安裝環境依賴（Go 1.23+ 與 Redis）

In [ ]:
# 1. 安裝 Redis 伺服器並在背景啟動
!apt-get update -qq && apt-get install -y -qq redis-server
!redis-server --daemonize yes
!redis-cli ping

# 2. 下載並設定 Go 1.23+ 編譯環境
import os
!wget -q https://go.dev/dl/go1.23.6.linux-amd64.tar.gz
!rm -rf /usr/local/go && tar -C /usr/local -xzf go1.23.6.linux-amd64.tar.gz
os.environ["PATH"] = "/usr/local/go/bin:" + os.environ["PATH"]
!go version

## 步驟 2：下載專案原始碼並編譯二進位檔案

In [ ]:
# 下載 Child-pi/ax 儲存庫
%cd /content
!rm -rf ax
!git clone https://github.com/Child-pi/ax.git
%cd /content/ax

# 相容處理 go.mod 中的版本標註
!sed -i 's/go 1.27.*/go 1.23/' go.mod

# 編譯 ax CLI、ax-server 控制平面服務與 ax-task-runner
!mkdir -p bin
!go build -trimpath -ldflags="-s -w" -o bin/ax ./cmd/ax
!go build -trimpath -ldflags="-s -w" -o bin/ax-server ./cmd/ax-server
!go build -trimpath -ldflags="-s -w" -o bin/ax-task-runner ./cmd/ax-task-runner

!ls -lh bin/

## 步驟 3：在背景啟動 AX API Server
`ax-server` 提供了 gRPC API 與 HTTP `/healthz` 探針，將所有宣告式物件存放在 Redis，並以 Redis Streams 作為事件佇列。

In [ ]:
import subprocess
import time

# 終止可能已存在的 ax-server
!pkill -f ax-server || true

# 在背景啟動 ax-server
server_proc = subprocess.Popen(
    ["./bin/ax-server", "--addr=:8080", "--redis-addr=localhost:6379"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(2)

# 測試健康檢查端點 (應顯示 ok)
!curl -s http://localhost:8080/healthz

## 步驟 4：撰寫宣告式 Manifest 並透過 `ax` CLI 套用
建立包含四大物件（`Task` + `Workspace` + `Gateway` + `Model`）的 YAML 定義檔：

In [ ]:
manifest_yaml = """apiVersion: ax.io/v1alpha1
kind: Task
metadata:
  name: colab-agent-task
  atespace: default
spec:
  command: ["python3", "-c", "print('Hello from sandboxed Agent!'); import time; time.sleep(10)"]
  resources:
    requests:
      cpu: "500m"
      memory: "1Gi"
    limits:
      cpu: "2"
      memory: "4Gi"
  workspaces:
    - name: agent-workspace
      path: "/content/workspace"
      goal: "Initialize Git repository and prepare build environment"
  gateway:
    name: agent-gateway
  env:
    - name: AGENT_MODE
      value: "colab_demo"
  debug: true
---
apiVersion: ax.io/v1alpha1
kind: Workspace
metadata:
  name: agent-workspace
  atespace: default
spec:
  git:
    - name: chalk
      repo: "https://github.com/chalk/chalk.git"
      branch: "main"
      depth: 1
  mcp:
    registries:
      - provider: google
        query: "mcp.tags:developer-tools"
---
apiVersion: ax.io/v1alpha1
kind: Gateway
metadata:
  name: agent-gateway
  atespace: default
spec:
  listeners:
    - name: grpc
      port: 8494
      protocol: gRPC
    - name: http
      port: 8080
      protocol: HTTP
  egress:
    allowlist:
      hosts:
        - host: "*"
          port: 443
---
apiVersion: ax.io/v1alpha1
kind: Model
metadata:
  name: default-model
  atespace: default
spec:
  provider: google
  model: gemini-3.8-flash
  parameters:
    temperature: 0.7
"""

with open("demo-agent.yaml", "w") as f:
    f.write(manifest_yaml)
print("demo-agent.yaml 寫入成功！")

### 套用 Manifest 並檢視資源清單

In [ ]:
# 設定 AX CLI 伺服器網址 (亦可使用 AX_SERVER 環境變數)
os.environ["AX_SERVER"] = "http://localhost:8080"

# 套用 YAML 定義檔
!./bin/ax apply -f demo-agent.yaml

# 查詢所有物件
print("\n--- 📋 Tasks ---")
!./bin/ax get tasks

print("\n--- 📂 Workspaces ---")
!./bin/ax get workspaces

print("\n--- 🌐 Gateways ---")
!./bin/ax get gateways

print("\n--- 🤖 Models ---")
!./bin/ax get models

### 檢視任務與工作區詳細規格

In [ ]:
!./bin/ax describe task colab-agent-task
!./bin/ax describe workspace agent-workspace

## 步驟 5：測試 Task Runner 沙盒執行器
`ax-task-runner` 是容器內的 PID 1 程序。它會：
1. 依據 Workspace 宣告拉取 Git 儲存庫
2. 啟動本機 Metadata Server（提供 Agent 查詢自身規格與設定）
3. 執行 Task 中指定的指令

In [ ]:
task_yaml = """apiVersion: ax.io/v1alpha1
kind: Task
metadata:
  name: colab-agent-task
  atespace: default
spec:
  command: ["bash", "-c", "echo 'Agent started! Checking workspace:'; ls -la /content/workspace/chalk; sleep 15"]
  workspaces:
    - name: agent-workspace
      path: "/content/workspace"
  env:
    - name: DEMO_RUN
      value: "true"
"""

workspace_yaml = """apiVersion: ax.io/v1alpha1
kind: Workspace
metadata:
  name: agent-workspace
  atespace: default
spec:
  git:
    - name: chalk
      repo: "https://github.com/chalk/chalk.git"
      branch: "main"
      depth: 1
"""

with open("/content/single-task.yaml", "w") as f:
    f.write(task_yaml)
with open("/content/single-workspace.yaml", "w") as f:
    f.write(workspace_yaml)
print("Task 與 Workspace 獨立檔案產生完成！")

In [ ]:
# 確保清理舊目錄
!rm -rf /content/workspace

# 啟動 ax-task-runner 在 port 8081
runner_proc = subprocess.Popen(
    ["./bin/ax-task-runner", "--port=8081", "--task-file=/content/single-task.yaml", "--workspace-file=/content/single-workspace.yaml"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# 等待初始化並檢查日誌
time.sleep(5)

# 測試 Runner 的健康探針與 Metadata 服務
print("Healthz 狀態:")
!curl -s http://localhost:8081/healthz

print("\nReadyz 狀態 (Workspace 是否就緒):")
!curl -s http://localhost:8081/readyz

print("\n從 Metadata API 讀取目前 Task 規格:")
!curl -s http://localhost:8081/metadata/v1alpha1/ax/task | head -n 25

print("\n檢查 Workspace 中的 Git Clone 成品:")
!ls -la /content/workspace/chalk

## 步驟 6：探索 Antigravity Bootstrap 與 Goal 引導設定
AX 的亮點之一是結合 **Google Antigravity SDK**。
當 `Task.workspaces[].goal` 被指定時，`ax-task-runner` 會自動調用 `antigravity_bootstrap.py`，
讓 Gemini Agent 自動理解任務目標，並在沙盒內部安裝對應的依賴套件（例如 Node.js, Python, Rust 或編譯工具鏈）。

In [ ]:
# 檢視 AX Task Runner 內建的 Antigravity 引導腳本
!head -n 50 cmd/ax-task-runner/antigravity_bootstrap.py

## 步驟 7：清理環境
停止背景執行的程序，並釋放資源。

In [ ]:
# 終止背景服務
try:
    server_proc.terminate()
    runner_proc.terminate()
except Exception:
    pass
print("✅ 演示完成，服務已停止！")